In [0]:
# DLT works with 3 types of datasets
# 1. Streaming Table (Permanent/Temporary) - Used as Append Data Sources, Incremental Data
# 2. Materialized Views - Used for transformations, aggregations or computations 
# 3. Views - Used for intermediate transformations, not stored in Target Schema
# DLT pipelines are powered by delta lake, so we get all the power of Delta Lake as well

import dlt

In [0]:
# Create a streaming table for Orders
# Delta table has property to work as both batch and stream mode.  
# by default name of function will be name of streaming table : orders_bronze 
@dlt.table(
    table_properties = {"quality" : "bronze"},
    comment = "Order bronze table"
)
def orders_bronze():
    df = spark.readStream.table("dbcertification.easewithdata.orders_raw")
    return df

In [0]:
# Create a Materialized View for Customers
# dlt.table will only be used for streaming table and materialized view 

@dlt.table(
    table_properties = {"quality" : "bronze"},
    comment = "Customer bronze table",
    name = "customer_bronze"
)
def cust_bronze():
    df = spark.read.table("dbcertification.easewithdata.customer_raw")
    return df

In [0]:
# Create a dlt view to join orders with customers
# LIVE keyword is used to reference DLT tables and views created in same pipeline

@dlt.view(
    comment = "Joined View"
)
def joined_vw():
    df_c = spark.read.table("LIVE.customer_bronze")
    df_o = spark.read.table("LIVE.orders_bronze")

    df_join = df_o.join(df_c, how = "left_outer", on = df_c.c_custkey == df_o.o_custkey)
      
    return df_join 


In [0]:
# Create MV to add new column
from pyspark.sql.functions import current_timestamp

@dlt.table(
    table_properties = {"quality" : "silver"},
    comment = "joined table",
    name = "joined_silver"
)
def joined_silver():
    df = spark.read.table("LIVE.joined_vw").withColumn("__insert_date",current_timestamp())
    return df

In [0]:
# Aggregate based on c_mktsegment and find the count of order  (c_orderkey)

from pyspark.sql.functions import current_timestamp, count

@dlt.table(
    table_properties = {"quality" : "gold"},
    comment = "Orders Aggregated table"
)
def orders_agg_gold():
    df = spark.read.table("LIVE.joined_silver")

    df_final = df.groupBy("c_mktsegment").agg(count("o_orderkey").alias("sum_orders")).withColumn("__insert_date",current_timestamp())
    
    return df_final